# **Install Python libraries**

In [0]:
# ============================================================
# BANGLAPAY | PHASE 0.4
# Install external source connectivity libraries
# ============================================================

%pip install --quiet confluent-kafka "psycopg[binary]" pymongo requests

# **Load configuration and secrets**

In [0]:
# ============================================================
# BANGLAPAY | SOURCE CONNECTIVITY CONFIGURATION
# ============================================================

import os
import uuid
import time
import json
import pandas as pd

ENVIRONMENT = "dev"  # Change to "prod" in PROD workspace

SCOPE_NAME = "banglapay-ingestion"

POSTGRES_DATABASE = "banglapay"
MONGODB_DATABASE = "banglapay_ops"

MAX_ROWS_PER_TABLE = 3
MAX_DOCUMENTS_PER_COLLECTION = 3
MAX_MESSAGES_PER_TOPIC = 2

# ------------------------------------------------------------
# Required secret keys
# ------------------------------------------------------------

REQUIRED_SECRETS = [
    "CONFLUENT_BOOTSTRAP_SERVERS",
    "CONFLUENT_API_KEY",
    "CONFLUENT_API_SECRET",

    "SCHEMA_REGISTRY_URL",
    "SCHEMA_REGISTRY_API_KEY",
    "SCHEMA_REGISTRY_API_SECRET",

    "NEON_JDBC_URL",
    "NEON_USERNAME",
    "NEON_PASSWORD",

    "MONGODB_CONNECTION_URI",
]

# ------------------------------------------------------------
# Verify scope and required keys
# ------------------------------------------------------------

available_keys = {
    item.key
    for item in dbutils.secrets.list(SCOPE_NAME)
}

missing_keys = [
    key for key in REQUIRED_SECRETS
    if key not in available_keys
]

if missing_keys:
    raise RuntimeError(
        f"Missing secret keys: {missing_keys}"
    )

# ------------------------------------------------------------
# Load secret values into notebook memory
# Never display this dictionary.
# ------------------------------------------------------------

SECRETS = {
    key: dbutils.secrets.get(
        scope=SCOPE_NAME,
        key=key
    )
    for key in REQUIRED_SECRETS
}

if any(not value.strip() for value in SECRETS.values()):
    raise RuntimeError("One or more required secrets are empty.")

# ------------------------------------------------------------
# Initialize test results
# ------------------------------------------------------------

results = []

def record_result(source, status, details):
    results.append({
        "environment": ENVIRONMENT.upper(),
        "source": source,
        "status": status,
        "details": details
    })

def safe_error(error):
    message = f"{type(error).__name__}: {error}"

    # Redact any exact secret values appearing in exceptions.
    for secret_value in SECRETS.values():
        if secret_value:
            message = message.replace(secret_value, "***")

    return message[:500]

print("Environment:", ENVIRONMENT.upper())
print("Secret scope:", SCOPE_NAME)
print("Required secret keys:", len(REQUIRED_SECRETS))
print("Secret verification: SUCCESS")

# **Confluent Schema Registry**

In [0]:
# ============================================================
# SOURCE 1: CONFLUENT SCHEMA REGISTRY
# Read subject metadata and latest schema versions.
# ============================================================

import requests
from urllib.parse import quote

try:
    registry_url = SECRETS["SCHEMA_REGISTRY_URL"].rstrip("/")

    response = requests.get(
        f"{registry_url}/subjects",
        auth=(
            SECRETS["SCHEMA_REGISTRY_API_KEY"],
            SECRETS["SCHEMA_REGISTRY_API_SECRET"]
        ),
        timeout=15
    )

    response.raise_for_status()

    subjects = response.json()

    schema_metadata = []

    for subject in subjects:

        latest_response = requests.get(
            f"{registry_url}/subjects/{quote(subject, safe='')}/versions/latest",
            auth=(
                SECRETS["SCHEMA_REGISTRY_API_KEY"],
                SECRETS["SCHEMA_REGISTRY_API_SECRET"]
            ),
            timeout=15
        )

        latest_response.raise_for_status()

        schema_info = latest_response.json()

        schema_metadata.append({
            "subject": subject,
            "version": schema_info.get("version"),
            "schema_type": schema_info.get("schemaType", "AVRO"),
            "schema_id": schema_info.get("id")
        })

    schema_df = pd.DataFrame(schema_metadata)

    print("Schema Registry: SUCCESS")
    print("Total subjects:", len(schema_metadata))

    display(schema_df)

    record_result(
        "Schema Registry",
        "PASS",
        f"{len(schema_metadata)} subjects accessible"
    )

except Exception as error:

    print("Schema Registry: FAILED")
    print(safe_error(error))

    record_result(
        "Schema Registry",
        "FAIL",
        safe_error(error)
    )

# **Confluent Kafka**

In [0]:
# ============================================================
# SOURCE 2: CONFLUENT KAFKA
# Read topic metadata and small message samples.
#
# Safety:
# - No messages produced.
# - New unique consumer group.
# - Auto commit disabled.
# - No offset commits.
# - Only two messages per topic.
# ============================================================

from confluent_kafka import (
    Consumer,
    KafkaError,
    KafkaException
)
from confluent_kafka.admin import AdminClient

kafka_config = {
    "bootstrap.servers":
        SECRETS["CONFLUENT_BOOTSTRAP_SERVERS"],

    "security.protocol": "SASL_SSL",
    "sasl.mechanism": "PLAIN",

    "sasl.username":
        SECRETS["CONFLUENT_API_KEY"],

    "sasl.password":
        SECRETS["CONFLUENT_API_SECRET"],

    "socket.timeout.ms": 10000,
    "request.timeout.ms": 10000
}

kafka_samples = []
kafka_topics = []

# ------------------------------------------------------------
# Step 1: List topics
# ------------------------------------------------------------

try:

    admin = AdminClient(kafka_config)

    metadata = admin.list_topics(timeout=15)

    kafka_topics = sorted([
        topic_name
        for topic_name, topic_metadata in metadata.topics.items()
        if not topic_name.startswith("_")
        and topic_metadata.error is None
    ])

    topic_metadata_rows = [
        {
            "topic": topic_name,
            "partitions": len(metadata.topics[topic_name].partitions)
        }
        for topic_name in kafka_topics
    ]

    display(pd.DataFrame(topic_metadata_rows))

    print("Kafka metadata: SUCCESS")
    print("Accessible topics:", len(kafka_topics))

    record_result(
        "Kafka Metadata",
        "PASS",
        f"{len(kafka_topics)} accessible topics"
    )

except Exception as error:

    print("Kafka metadata: FAILED")
    print(safe_error(error))

    record_result(
        "Kafka Metadata",
        "FAIL",
        safe_error(error)
    )

# ------------------------------------------------------------
# Step 2: Consume small samples from all accessible topics
# ------------------------------------------------------------

if kafka_topics:

    consumer_config = {
        **kafka_config,

        "group.id": (
            f"banglapay-phase0-{ENVIRONMENT}-{uuid.uuid4().hex}"
        ),

        "auto.offset.reset": "earliest",
        "enable.auto.commit": False,
        "enable.auto.offset.store": False
    }

    consumer = Consumer(consumer_config)

    topic_counts = {
        topic: 0 for topic in kafka_topics
    }

    try:

        consumer.subscribe(kafka_topics)

        deadline = time.monotonic() + 18

        while time.monotonic() < deadline:

            if all(
                count >= MAX_MESSAGES_PER_TOPIC
                for count in topic_counts.values()
            ):
                break

            message = consumer.poll(1.0)

            if message is None:
                continue

            if message.error():

                if message.error().code() == KafkaError._PARTITION_EOF:
                    continue

                raise KafkaException(message.error())

            topic = message.topic()

            if topic_counts.get(topic, 0) >= MAX_MESSAGES_PER_TOPIC:
                continue

            key_bytes = message.key() or b""
            value_bytes = message.value() or b""

            kafka_samples.append({
                "topic": topic,
                "partition": message.partition(),
                "offset": message.offset(),
                "key_bytes": len(key_bytes),
                "value_bytes": len(value_bytes),
                "value_prefix_hex": value_bytes[:16].hex()
            })

            topic_counts[topic] += 1

        print("Kafka sample consumption: COMPLETED")

        display(
            pd.DataFrame([
                {
                    "topic": topic,
                    "sample_messages": topic_counts[topic]
                }
                for topic in kafka_topics
            ])
        )

        display(pd.DataFrame(kafka_samples))

        record_result(
            "Kafka Message Read",
            "PASS",
            f"Read {len(kafka_samples)} messages across "
            f"{len(kafka_topics)} topics; "
            f"zero-message topics: "
            f"{sum(v == 0 for v in topic_counts.values())}"
        )

    except Exception as error:

        print("Kafka message read: FAILED")
        print(safe_error(error))

        record_result(
            "Kafka Message Read",
            "FAIL",
            safe_error(error)
        )

    finally:
        consumer.close()

else:

    record_result(
        "Kafka Message Read",
        "SKIPPED",
        "No accessible topics found"
    )

# **Neon PostgreSQL**

In [0]:
# ============================================================
# SOURCE 3: NEON POSTGRESQL
#
# Read-only connection.
# Discover all user tables.
# Preview maximum three rows from every table.
# ============================================================

import psycopg
from psycopg import sql

try:

    jdbc_url = SECRETS["NEON_JDBC_URL"].strip()

    # Convert JDBC URL to PostgreSQL URI for psycopg.
    if jdbc_url.startswith("jdbc:"):
        pg_url = jdbc_url[len("jdbc:"):]

    elif jdbc_url.startswith(
        ("postgresql://", "postgres://")
    ):
        pg_url = jdbc_url

    else:
        raise ValueError(
            "NEON_JDBC_URL must be a JDBC PostgreSQL URL "
            "or PostgreSQL URI."
        )

    pg_summary = []

    with psycopg.connect(
        pg_url,
        user=SECRETS["NEON_USERNAME"],
        password=SECRETS["NEON_PASSWORD"],
        connect_timeout=10,
        sslmode="require"
    ) as conn:

        with conn.cursor() as cursor:

            # Prevent any accidental write operation.
            cursor.execute("SET TRANSACTION READ ONLY")

            # Discover all user-created tables.
            cursor.execute("""
                SELECT
                    table_schema,
                    table_name
                FROM information_schema.tables
                WHERE table_type = 'BASE TABLE'
                  AND table_schema NOT IN (
                      'pg_catalog',
                      'information_schema'
                  )
                ORDER BY
                    table_schema,
                    table_name
            """)

            tables = cursor.fetchall()

        print("PostgreSQL connection: SUCCESS")
        print("Discovered tables:", len(tables))

        # ----------------------------------------------------
        # Read a small preview from every table.
        # ----------------------------------------------------

        SENSITIVE_HINTS = (
            "password",
            "secret",
            "token",
            "email",
            "phone",
            "mobile",
            "msisdn",
            "first_name",
            "last_name",
            "customer_name",
            "address",
            "national_id",
            "nid",
            "date_of_birth"
        )

        for schema_name, table_name in tables:

            with conn.cursor() as cursor:

                query = sql.SQL(
                    "SELECT * FROM {}.{} LIMIT %s"
                ).format(
                    sql.Identifier(schema_name),
                    sql.Identifier(table_name)
                )

                cursor.execute(
                    query,
                    (MAX_ROWS_PER_TABLE,)
                )

                rows = cursor.fetchall()

                columns = [
                    item.name for item in cursor.description
                ]

                preview_df = pd.DataFrame(
                    rows,
                    columns=columns
                )

                # Mask obvious personal/sensitive columns.
                for column in columns:

                    column_name = column.lower()

                    if any(
                        hint in column_name
                        for hint in SENSITIVE_HINTS
                    ):
                        preview_df[column] = "***MASKED***"

            pg_summary.append({
                "schema": schema_name,
                "table": table_name,
                "sample_rows": len(rows),
                "status": "SUCCESS"
            })

            print(
                f"Table: {schema_name}.{table_name} "
                f"| Sample rows: {len(rows)}"
            )

            if not preview_df.empty:
                display(preview_df)

    display(pd.DataFrame(pg_summary))

    record_result(
        "Neon PostgreSQL",
        "PASS",
        f"Discovered and sampled {len(tables)} user tables"
    )

except Exception as error:

    print("Neon PostgreSQL: FAILED")
    print(safe_error(error))

    record_result(
        "Neon PostgreSQL",
        "FAIL",
        safe_error(error)
    )

# **MongoDB Atlas**

In [0]:
# ============================================================
# SOURCE 4: MONGODB ATLAS
#
# Discover all collections.
# Preview maximum three documents per collection.
# No insert, update or delete operations.
# ============================================================

from pymongo import MongoClient
from bson import json_util

mongo_client = None

try:

    mongo_client = MongoClient(
        SECRETS["MONGODB_CONNECTION_URI"],
        serverSelectionTimeoutMS=10000,
        connectTimeoutMS=10000,
        socketTimeoutMS=15000
    )

    # Validate authentication and connectivity.
    mongo_client.admin.command("ping")

    mongo_db = mongo_client[MONGODB_DATABASE]

    collection_names = sorted(
        mongo_db.list_collection_names()
    )

    print("MongoDB connection: SUCCESS")
    print("Discovered collections:", len(collection_names))

    mongo_summary = []

    for collection_name in collection_names:

        collection = mongo_db[collection_name]

        documents = list(
            collection.find({}).limit(
                MAX_DOCUMENTS_PER_COLLECTION
            )
        )

        # Convert BSON values into JSON-safe structures.
        safe_documents = [
            json.loads(json_util.dumps(document))
            for document in documents
        ]

        collection_df = pd.json_normalize(
            safe_documents
        )

        SENSITIVE_HINTS = (
            "password",
            "secret",
            "token",
            "email",
            "phone",
            "mobile",
            "msisdn",
            "customer_name",
            "address",
            "national_id",
            "nid",
            "date_of_birth"
        )

        # Mask obvious sensitive fields.
        for column in list(collection_df):

            column_name = column.lower()

            if any(
                hint in column_name
                for hint in SENSITIVE_HINTS
            ):
                collection_df[column] = "***MASKED***"

        mongo_summary.append({
            "collection": collection_name,
            "sample_documents": len(documents),
            "status": "SUCCESS"
        })

        print(
            f"Collection: {collection_name} "
            f"| Sample documents: {len(documents)}"
        )

        if not collection_df.empty:
            display(collection_df)

    display(pd.DataFrame(mongo_summary))

    record_result(
        "MongoDB Atlas",
        "PASS",
        f"Discovered and sampled {len(collection_names)} collections"
    )

except Exception as error:

    print("MongoDB Atlas: FAILED")
    print(safe_error(error))

    record_result(
        "MongoDB Atlas",
        "FAIL",
        safe_error(error)
    )

finally:

    if mongo_client is not None:
        try:
            mongo_client.close()
        except Exception:
            pass

# **Final connectivity report**

In [0]:
# ============================================================
# BANGLAPAY | PHASE 0.4
# Final Source Connectivity Report
# ============================================================

final_report = pd.DataFrame(results)

display(final_report)

passed = sum(
    result["status"] == "PASS"
    for result in results
)

failed = sum(
    result["status"] == "FAIL"
    for result in results
)

print("=" * 55)
print("BANGLAPAY SOURCE CONNECTIVITY REPORT")
print("=" * 55)

print("Environment:", ENVIRONMENT.upper())
print("Tests passed:", passed)
print("Tests failed:", failed)
print("Total tests:", len(results))

if failed == 0:
    print("OVERALL STATUS: SUCCESS")
else:
    print("OVERALL STATUS: PARTIAL / FAILED")